# Build 1: LoRA fine-tuning, end to end

**Goal:** understand every step of fine-tuning Llama-2-7B on SQuAD with LoRA,
on a *tiny* model so it runs in seconds.

**What is LoRA?** We freeze the base weight $W$ and learn a small low-rank update:

$$W' = W + \frac{\alpha}{r}\, B A,\quad A\in\mathbb{R}^{d\times r},\; B\in\mathbb{R}^{r\times d},\; r\ll d$$

Only $A,B$ train. With $r=80,\ \alpha=160$ this matches HiMoLE's trainable-param budget.

> Cells that touch a `# TODO` will raise `NotImplementedError` until you fill it.
> That is expected — the notebook shows you *where* each learning point lives.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))  # repo root onto the path


In [2]:
from himole.config import BaselineConfig

cfg = BaselineConfig()
cfg.use_tiny = True          # sshleifer/tiny-gpt2, runs on CPU
cfg.max_train_samples = 8    # keep it tiny
cfg.max_steps = 2
cfg

BaselineConfig(base_model='meta-llama/Llama-2-7b-hf', tiny_model='sshleifer/tiny-gpt2', use_tiny=True, lora_r=80, lora_alpha=160, lora_dropout=0.05, target_modules=('up_proj', 'down_proj', 'gate_proj'), id_dataset='rajpurkar/squad', cutoff_len=1024, max_train_samples=8, lr=0.0003, batch_size=16, max_steps=2, eval_every=1, early_stop_patience=10, seed=42, output_dir='outputs/baseline_lora')

## Step 1 — Data & the loss mask

We turn a SQuAD example into `input_ids` (prompt + answer) and `labels`.
The trick: set `labels = -100` on every **prompt** token so the loss only
scores the **answer**. (PyTorch cross-entropy ignores `-100`.)

**TODO sites:** `build_prompt` and `format_example` in `himole/data/squad.py`.

In [3]:
from transformers import AutoTokenizer
from himole.data.squad import build_prompt, format_example

tok = AutoTokenizer.from_pretrained(cfg.tiny_model)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

ex = {'context': 'Paris is the capital of France.',
      'question': 'What is the capital of France?',
      'answers': {'text': ['Paris']}}

out = format_example(ex, tok, cutoff_len=64)   # raises until you fill the TODO
for tid, lab in zip(out['input_ids'], out['labels']):
    print(f"{tid:>6}  label={lab}")           # label=-100 means 'masked / no loss'

/home/zzz010122/anaconda3/envs/AIplayground/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


 33706  label=-100
   262  label=-100
  1808  label=-100
  1262  label=-100
   262  label=-100
  4732  label=-100
    13  label=-100
   198  label=-100
 21947  label=-100
    25  label=-100
  6342  label=-100
   318  label=-100
   262  label=-100
  3139  label=-100
   286  label=-100
  4881  label=-100
    13  label=-100
   198  label=-100
 24361  label=-100
    25  label=-100
  1867  label=-100
   318  label=-100
   262  label=-100
  3139  label=-100
   286  label=-100
  4881  label=-100
    30  label=-100
   198  label=-100
 33706  label=-100
    25  label=-100
  6342  label=6342
 50256  label=50256


## Step 2 — Attach LoRA

Freeze the base model, add LoRA adapters on the FFN projections.
`print_trainable_parameters()` shows how tiny the trainable slice is.

**TODO site:** `attach_lora` in `himole/model/baseline_lora.py`.

In [4]:
from himole.model.baseline_lora import load_base_model, attach_lora

model = attach_lora(load_base_model(cfg), cfg)   # raises until you fill the TODO

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 29/29 [00:00<00:00, 14642.45it/s]
The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: sshleifer/tiny-gpt2
Key                                   | Status     |  | 
--------------------------------------+------------+--+-
transformer.h.{0, 1}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


trainable params: 3,840 || all params: 207,068 || trainable%: 1.8545


/home/zzz010122/anaconda3/envs/AIplayground/lib/python3.14/site-packages/peft/tuners/lora/layer.py:2504: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


## Step 3 — One training step

The whole loop is just: `outputs = model(**batch)` → `loss = outputs.loss`
→ `loss.backward()` → `optimizer.step()` → `optimizer.zero_grad()`.

**TODO site:** the core step in `himole/train/loop.py`.

In [5]:
from himole.data.squad import load_squad
from himole.train.loop import train

train_ds, _ = load_squad(tok, cfg)
id_pairs = [{'prompt': build_prompt(ex['context'], ex['question']),
             'gold': ex['answers']['text'][0]}]

train(model, tok, train_ds, id_pairs, cfg)       # raises until TODOs are filled

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


{'best_em': 0.0, 'last': {'em': 0.0, 'rouge2': 0.0}}

## Step 4 — Evaluate EM / ROUGE-2

Generate answers and score them. **TODO sites:** `himole/eval/metrics.py`.

In [6]:
from himole.eval.generate import evaluate

evaluate(model, tok, id_pairs)                   # raises until metrics TODOs are filled

{'em': 0.0, 'rouge2': 0.0}

## Your homework — fill the TODOs in this order

Each one has a test that turns green when you get it right:

1. `himole/data/squad.py` → `build_prompt`, `format_example`  —  `pytest tests/test_data_masking.py`
2. `himole/eval/metrics.py` → normalize / EM / ROUGE-2 / score_batch  —  `pytest tests/test_metrics.py`
3. `himole/model/baseline_lora.py` → `attach_lora` (the `LoraConfig`)
4. `himole/train/loop.py` → the forward/backward/step
5. `himole/data/newsqa.py` → the OOD loader

Then run the smoke test end-to-end:
`python scripts/train_baseline.py --tiny --steps 5 --samples 8`